# Wandas Pipeline Recipes

This notebook checks the current Recipe UX with small synthetic frames. It does not download data or require optional heavy dependencies.

The examples cover:

- extracting a linear `RecipeSpec` from exploratory frame method chains,
- replaying the recipe on another frame,
- extracting a graph recipe from two frame inputs,
- replaying an importable custom function,
- adding an explicit terminal step,
- seeing `RecipeExtractionError` for an unsupported lambda boundary.

## Create Small Sample Frames

In [ ]:
import numpy as np

from wandas.frames.channel import ChannelFrame
from wandas.pipeline import GraphRecipeSpec, OperationSpec, RecipeExtractionError, RecipeSpec, TerminalStep

sampling_rate = 8000
time = np.linspace(0.0, 0.25, int(sampling_rate * 0.25), endpoint=False)

source_data = (0.25 + np.sin(2 * np.pi * 440 * time)).reshape(1, -1)
other_data = (0.10 + 0.5 * np.sin(2 * np.pi * 880 * time)).reshape(1, -1)

frame = ChannelFrame.from_numpy(source_data, sampling_rate=sampling_rate, label="source")
other_frame = ChannelFrame.from_numpy(other_data, sampling_rate=sampling_rate, label="other")

frame.shape, other_frame.shape

## Explore With Frame Methods, Then Extract A Recipe

The exploratory code stays normal Wandas code. `RecipeSpec.from_frame(...)` reads the processed frame lineage and converts it to a replayable linear recipe.

In [ ]:
processed = frame.remove_dc().trim(start=0.02, end=0.20).normalize()

recipe = RecipeSpec.from_frame(processed)
recipe.to_dict()

In [ ]:
replayed = recipe.apply(other_frame)

print(type(replayed).__name__)
print([record["operation"] for record in replayed.operation_history])
print(replayed.shape)

## GraphRecipeSpec For Two Frame Inputs

`RecipeSpec` is intentionally linear and single-input. Use `GraphRecipeSpec` when the calculation has two frame parents and one binary merge.

In [ ]:
signal = ChannelFrame.from_numpy(source_data, sampling_rate=sampling_rate, label="signal")
noise = ChannelFrame.from_numpy(other_data * 0.1, sampling_rate=sampling_rate, label="noise")

mixed = (signal.remove_dc() + noise.normalize()).trim(start=0.0, end=0.10)

graph_recipe = GraphRecipeSpec.from_frame(mixed, input_names=("signal", "noise"))
graph_replayed = graph_recipe.apply({"signal": signal, "noise": noise})

print(graph_recipe.to_dict())
print(np.allclose(graph_replayed.data, mixed.data))

## Custom Function Recipe

Custom functions must be importable module-level functions. A notebook cell function is usually `__main__`, so this example creates a tiny temporary module to show the supported path.

In [ ]:
import importlib
import sys
import tempfile
from pathlib import Path

module_dir = Path(tempfile.mkdtemp(prefix="wandas_recipe_ops_"))
sys.path.insert(0, str(module_dir))

(module_dir / "recipe_custom_ops.py").write_text(
    "def scale(data, gain):\n"
    "    return data * gain\n\n"
    "def same_shape(shape):\n"
    "    return shape\n",
    encoding="utf-8",
)

recipe_custom_ops = importlib.import_module("recipe_custom_ops")

custom_processed = frame.apply(
    recipe_custom_ops.scale,
    output_shape_func=recipe_custom_ops.same_shape,
    gain=0.5,
)

custom_recipe = RecipeSpec.from_frame(custom_processed)
custom_replayed = custom_recipe.apply(other_frame)

print(custom_recipe.to_dict())
print(custom_replayed.shape)

## Explicit Terminal Step

Terminal values such as `rms` return arrays, not frames. They do not carry lineage, so terminal steps are explicit recipe steps.

In [ ]:
terminal_recipe = RecipeSpec(
    [
        OperationSpec("remove_dc"),
        TerminalStep("rms"),
    ]
)

rms_values = terminal_recipe.apply(frame)
rms_values

## Expected Boundary: Lambda Custom Function

`RecipeExtractionError` is how Wandas makes an unsupported replay boundary explicit. The frame calculation can run, but the recipe extractor refuses to serialize a lambda because replay would not be importable.

In [ ]:
bad = frame.apply(
    lambda data, gain: data * gain,
    output_shape_func=lambda shape: shape,
    gain=2.0,
)

try:
    RecipeSpec.from_frame(bad)
except RecipeExtractionError as exc:
    print(type(exc).__name__)
    print(str(exc).splitlines()[0])